In [2]:
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# 1. Initialize the Professional Tokenizer (GPT-2 standard)
tokenizer = tiktoken.get_encoding('gpt2')
vocab_size = tokenizer.n_vocab  # Jumps from ~89 to 50,257!

print(f"Vocabulary Size upgraded to: {vocab_size}")

# 2. Reload and Re-encode Data
device = "cuda" if torch.cuda.is_available() else "cpu"
data_dir = "data/data.txt"

# Make sure you have a data.txt file
try:
    text = open(data_dir, 'r').read()
    # ENCODE: Convert text to BPE tokens (List of integers)
    encoded_text = tokenizer.encode(text)
    data = torch.tensor(encoded_text, dtype=torch.long, device=device)
    print(f"Data re-encoded. Total tokens: {len(data)}")
except FileNotFoundError:
    print("Error: data.txt not found. Please upload a text file named 'data.txt'.")
    # Fallback for demonstration
    data = torch.tensor(tokenizer.encode("Hello world " * 1000), dtype=torch.long, device=device)

# 3. Update Helper Functions to use new Tokenizer
def encode(input_text: str):
    return tokenizer.encode(input_text)

def decode(input_tokens: list):
    return tokenizer.decode(input_tokens)

# 4. IMPORTANT: Re-create DataLoaders with new data
# (We need the DataLoader class from Part 1. I will redefine it briefly to ensure this cell runs)
class DataLoader:
    def __init__(self, tokens, batch_size, context_length) -> None:
        self.tokens = tokens
        self.batch_size = batch_size
        self.context_length = context_length
        self.current_position = 0

    def get_batch(self):
        b, c = self.batch_size, self.context_length
        if self.current_position + b * c + 1 >= len(self.tokens):
            self.current_position = 0
        
        d = self.tokens[self.current_position : self.current_position + b * c + 1]
        x = (d[:-1]).view(b, c)
        y = (d[1:]).view(b, c)
        self.current_position += b * c 
        return x, y

# Split Data
n_data = len(data)
train_data = data[:int(n_data * 0.9)]
eval_data = data[int(n_data * 0.9):]

# Update Hyperparameters for the Final Build
train_batch_size = 16 
context_length = 512 # Increased context

train_loader = DataLoader(train_data, train_batch_size, context_length)
eval_loader = DataLoader(eval_data, train_batch_size, context_length)

print("Tokenizer upgraded and DataLoaders refreshed.")

Vocabulary Size upgraded to: 50257
Data re-encoded. Total tokens: 232724
Tokenizer upgraded and DataLoaders refreshed.


In [4]:
# 5. Update Positional Encoding to handle larger d_model sizes
class PositionalEncoding(nn.Module):
    def __init__(self, context_length, d_model) -> None:
        super().__init__()
        # 1. Create a blank matrix of shape (context_length, d_model)
        pe = torch.zeros(context_length, d_model)
        
        # 2. Create a vector with positions [0, 1, 2, ..., context_length-1]
        position = torch.arange(0, context_length, dtype=torch.float).unsqueeze(1)
        
        # 3. Create the divisor terms (frequencies)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 4. Compute Sine and Cosine patterns
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # FIX: Handle odd d_model sizes (like 89) by slicing the cosine result
        cos_part = torch.cos(position * div_term)
        pe[:, 1::2] = cos_part[:, :pe[:, 1::2].size(1)]
        
        pe = pe.unsqueeze(0)  # Shape: (1, context_length, d_model)
        
        # Register as a buffer (not a trainable parameter)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Add the positional encodings to the input embeddings
        return x + self.pe[:, :x.size(1), :]

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Configuration ---
n_heads = 4  # We split our 512 dimensions into 4 heads of 128 dims each

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Validation: The total model size must be cleanly divisible by the number of heads
        assert (n_heads * self.head_dim == d_model)

        # The Projections
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        
        # Final aggregation
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.2)

    def forward(self, inputs: torch.Tensor):
        B, seq_length, d_model = inputs.shape
        
        # 1. Project inputs into Q, K, and V
        # Reshape to: (Batch, Seq_Len, n_heads, head_dim) -> Then Swap to: (Batch, n_heads, Seq_Len, head_dim)
        # This separates the "Heads" so they can work in parallel
        Q = self.query(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.key(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.value(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        
        # 2. Compute attention scores
        # We divide by sqrt(head_dim) to keep gradients stable (Scaled Dot-Product Attention)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # 3. Apply Mask (The Time Shield)
        mask = torch.triu(torch.ones(seq_length, seq_length), diagonal=1).bool().to(inputs.device)
        attention_scores = attention_scores.masked_fill(mask, float('-inf'))
        
        # 4. Softmax & Dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # 5. Aggregate Values
        attention_output = torch.matmul(attention_weights, V)

        # 6. Concatenate heads and restore original shape
        # Swap back: (Batch, Seq_Len, n_heads, head_dim) -> Flatten: (Batch, Seq_Len, d_model)
        attention_output = attention_output.permute(0, 2, 1, 3).contiguous()
        attention_output = attention_output.view(B, seq_length, d_model)

        # 7. Final Linear Transformation
        out = self.fc_out(attention_output)
        
        return out

print("Multi-Head Attention Module Online.")

Multi-Head Attention Module Online.


In [6]:
# --- Cell: GPT Decoder Block ---
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # 1. The Attention Head (The "Brain" of this floor)
        self.att = MultiHeadAttention(d_model, n_heads)
        self.ln1 = nn.LayerNorm(d_model)
        
        # 2. The Feed Forward Network (The "Processing Unit" of this floor)
        self.fcn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.2)

    def forward(self, logits):
        # Step 1: Attention + Residual + Norm
        # We calculate attention, ADD it to the original input (Residual), and Normalize
        att_logits = self.att(logits)
        adn_logits = self.ln1(logits + att_logits)
        
        logits = self.dropout(adn_logits)
        
        # Step 2: Feed Forward + Residual + Norm
        # We process the data, ADD it to the previous step (Residual), and Normalize
        logits = self.fcn(logits)
        logits = self.ln2(logits + adn_logits)
        
        return logits

print("GPTBlock defined. Ready for stacking.")

GPTBlock defined. Ready for stacking.


In [7]:
# --- Cell 2: Final GPT Architecture ---
d_model = 512
n_layers = 1  # As per tutorial (approx 29M params due to large vocab)
n_heads = 4

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model) # Word Token Embeddings
        self.wpe = PositionalEncoding(context_length, d_model) # Word Position Encodings
        
        # The Stack of Decoder Blocks
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads) for _ in range(n_layers)])
        
        # Final Projection
        self.linear1 = nn.Linear(d_model, vocab_size)
        
        # NEW: Parameter Sharing
        # We physically force the weights to be the same memory object
        self.wte.weight = self.linear1.weight

    def forward(self, inputs, targets=None):
        logits = self.wte(inputs)
        logits = self.wpe(logits)
        
        for block in self.blocks:
            logits = block(logits)
        
        logits = self.linear1(logits)
        
        loss = None
        if targets is not None:
            batch_size, sequence_length, vocab_size = logits.shape
            logits = logits.view(batch_size * sequence_length, vocab_size)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        output = inputs.clone()
        for _ in range(max_new_tokens):
            if inputs.size(1) > context_length:
                inputs = inputs[:, -context_length:]
            logits, _ = self(inputs)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat([inputs, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
        return [decode(out.tolist()) for out in output]

# Initialize and Compile
m = GPT(vocab_size=vocab_size, d_model=d_model, n_heads=n_heads, n_layers=n_layers).to(device)

# Optional: Torch Compile (Turbo Mode)
# This merges operations for speed. Requires PyTorch 2.0+
try:
    m = torch.compile(m)
    print("Model Compiled with torch.compile (Turbo Mode ON)")
except:
    print("torch.compile skipped (Standard Mode)")

print(f"Final Model Initialized. Parameters: {sum(p.numel() for p in m.parameters())/1e6:.1f}M")

Model Compiled with torch.compile (Turbo Mode ON)
Final Model Initialized. Parameters: 28.9M


In [9]:
# --- Cell 3: Final Training Loop ---
import time

# Hyperparameters
lr = 1e-3
epochs = 1000       #3500 As per tutorial
eval_steps = 100

# Optimizer & Scheduler
optim = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.1)
# Cosine Decay: Smoothly lowers LR from 1e-3 to 1e-4 over 3000 steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=3000, eta_min=lr*0.1)

train_loss = {}

print("Starting Final Training Run...")
start_time = time.time()

for e in range(epochs):
    # 1. Get Data
    xb, yb = train_loader.get_batch()

    # 2. Forward & Backward
    logits, loss = m(xb, yb)
    optim.zero_grad(set_to_none=True)
    loss.backward()

    # 3. NEW: Gradient Clipping (The Safety Valve)
    # Prevents gradients from getting too large and destabilizing training
    torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1)

    # 4. Step
    optim.step()
    scheduler.step() # Update the learning rate
    
    train_loss[e] = loss.item()

    # 5. Evaluation & Logging
    if e % eval_steps == 0 or e == epochs-1:
        m.eval()
        with torch.no_grad():
            xvb, yvb = eval_loader.get_batch()
            _, e_loss = m(xvb, yvb)
        
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch: {e}\tLR: {current_lr:.5f}\tTrain Loss: {loss.item():.4f}\tEval Loss: {e_loss:.4f}")
        m.train()

print(f"Training Finished in {(time.time()-start_time)/60:.1f} minutes.")


Starting Final Training Run...
Epoch: 0	LR: 0.00100	Train Loss: 6.3568	Eval Loss: 6.3090


KeyboardInterrupt: 

In [1]:
print(m)
print(f"Total Parameters: {round(sum(p.numel() for p in m.parameters() if p. requires_grad) / 1000000) }M")

NameError: name 'm' is not defined

In [ ]:

# --- Final Generation Test ---
print("\n--- Final Model Output ---")
start_context = "Once upon a time"
encoded_start = torch.tensor(encode(start_context), dtype=torch.long, device=device).unsqueeze(0)
generated_output = m.generate(encoded_start, max_new_tokens=50)
print(generated_output[0])